# DataLoaderのチューニング

---
## 目的
`torch.utils.data.DataLoader`の`num_workers`，`pin_memory`，`collate_fn`という3つの引数の役割を理解し，学習速度の改善やデータ形式の違いへの対応方法を身につける．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import time

import torch
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## num_workers：データ読み込みの並列化

`DataLoader`は，デフォルト（`num_workers=0`）では，メインプロセスがバッチの作成とネットワークの計算を順番に行います．
そのため，画像の読み込みやData Augmentationなどのデータ準備に時間がかかる場合，GPUがネットワークの計算を待っている間もCPU側の処理を待たなければならず，全体の学習時間が長くなってしまうことがあります．

`num_workers`に1以上の値を指定すると，指定した数のサブプロセスがバックグラウンドでデータの準備を並列に行うようになります．これにより，あるバッチをGPUで計算している間に，次のバッチの準備をCPU側で進めておくことができ，データ準備がボトルネックになっている場合には学習速度が改善します．

### 実験用データセットの準備

`num_workers`の効果を分かりやすく確認するために，1サンプルを取得するたびにわずかな待ち時間（`time.sleep`）が発生するダミーのDatasetを用意します．
これは，実際のディスクからの画像読み込みや，重いData Augmentation処理にかかる時間を模したものです．

In [ ]:
class SlowMNIST(data.Dataset):
    def __init__(self, base_dataset, delay=0.002):
        self.base_dataset = base_dataset
        self.delay = delay

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        time.sleep(self.delay)  ### データ準備の重い処理を模したダミーの待ち時間
        return self.base_dataset[idx]

mnist_train = torchvision.datasets.MNIST(root="./", train=True, transform=transforms.ToTensor(), download=True)
slow_train_data = SlowMNIST(torch.utils.data.Subset(mnist_train, range(2000)))

### num_workersを変えて所要時間を比較

`num_workers`を`0, 2, 4`と変化させて，データセットを1周（1エポック分）読み込むのにかかる時間を比較します．

In [ ]:
num_workers_list = [0, 2, 4]
elapsed_times = []

for num_workers in num_workers_list:
    loader = data.DataLoader(slow_train_data, batch_size=64, shuffle=True, num_workers=num_workers)

    start = time.time()
    for image, label in loader:
        pass
    elapsed = time.time() - start

    elapsed_times.append(elapsed)
    print("num_workers: {}, elapsed time: {:.2f} sec".format(num_workers, elapsed))

In [ ]:
plt.bar([str(n) for n in num_workers_list], elapsed_times)
plt.xlabel('num_workers')
plt.ylabel('elapsed time [sec]')
plt.title('num_workersごとの1エポックあたりの所要時間')
plt.show()

`num_workers`を増やすほど所要時間が短くなっていることが確認できます．
ただし，`num_workers`を増やしすぎると，サブプロセスの起動や管理に伴うオーバーヘッドが大きくなったり，CPUのコア数を超えるとかえって性能が低下したりする場合があります．一般的には，CPUのコア数を目安に，実際に速度を計測しながら適切な値を決定します．

## pin_memory：GPUへの転送の高速化

CPU上のメモリは通常，OSによって物理メモリと仮想メモリの間でページの入れ替え（スワップ）が行われる「ページング可能（pageable）」な領域に確保されます．このような領域からGPUへデータを転送する際には，一度OSが管理しない「ページ固定（pinned / page-locked）」領域へコピーしてから転送する必要があり，余分な処理が発生します．

`DataLoader`に`pin_memory=True`を指定すると，バッチデータがあらかじめページ固定領域に確保されるため，GPUへの転送（`.to(device)`）がより高速になります．GPUを利用する場合は，基本的に`pin_memory=True`を指定することが推奨されます．

In [ ]:
### ページ固定メモリと通常のメモリからのGPU転送時間を比較
data_size = (256, 3, 224, 224)
n_iters = 20

pageable_tensor = torch.randn(*data_size)
pinned_tensor = torch.randn(*data_size).pin_memory()

if device.type == 'cuda':
    ### pageableなメモリからの転送
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_iters):
        _ = pageable_tensor.to(device, non_blocking=True)
    torch.cuda.synchronize()
    pageable_time = time.time() - start

    ### pinnedなメモリからの転送
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(n_iters):
        _ = pinned_tensor.to(device, non_blocking=True)
    torch.cuda.synchronize()
    pinned_time = time.time() - start

    print("pageable memory -> GPU: {:.4f} sec".format(pageable_time))
    print("pinned memory   -> GPU: {:.4f} sec".format(pinned_time))
else:
    print("GPUが利用できないため，転送時間の比較はスキップします．")

## collate_fn：バッチの作成方法のカスタマイズ

`DataLoader`は，`Dataset`の`__getitem__`が返すサンプルを1つのバッチにまとめる際に，内部で`collate_fn`という関数を使用しています．デフォルトの`collate_fn`は，すべてのサンプルが同じ形状であることを前提に，テンソルを`torch.stack`でまとめます．

しかし，例えば系列データのように，サンプルごとにデータの長さ（系列長）が異なる場合，そのままでは1つのテンソルにまとめることができません．このような場合には，`collate_fn`を自作して`DataLoader`に渡すことで，バッチの作成方法を独自に定義できます．
以下では，サンプルごとに長さが異なるダミーの系列データに対して，バッチ内で最も長い系列にあわせて0でパディング（padding）する`collate_fn`を実装します．

In [ ]:
class VariableLengthDataset(data.Dataset):
    def __init__(self, num_samples=10):
        ### サンプルごとにランダムな長さ（3〜8）の系列データを生成
        self.data = [torch.randn(torch.randint(3, 9, (1,)).item(), 5) for _ in range(num_samples)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def padding_collate_fn(batch):
    lengths = [sample.size(0) for sample in batch]
    max_len = max(lengths)

    ### 最大系列長にあわせて0でパディングしたテンソルを作成
    padded_batch = torch.zeros(len(batch), max_len, batch[0].size(1))
    for i, sample in enumerate(batch):
        padded_batch[i, :sample.size(0)] = sample

    return padded_batch, torch.tensor(lengths)

var_len_dataset = VariableLengthDataset(num_samples=10)
var_len_loader = data.DataLoader(var_len_dataset, batch_size=4, shuffle=False, collate_fn=padding_collate_fn)

for padded_batch, lengths in var_len_loader:
    print("padded_batch.shape:", padded_batch.shape, ", lengths:", lengths)

## 課題

1. `SlowMNIST`の`delay`や`num_workers`の値を変更し，`num_workers`を増やしても速度が改善しなくなる，あるいは悪化するポイントがないか確認してみましょう．

2. `padding_collate_fn`を変更し，パディングする代わりに，各サンプルをテンソルにまとめずリストのまま返す`collate_fn`を実装してみましょう．